# Positional Encoding: Sinusoidal vs RoPE

Companion notebook for the [Positional Encoding](https://codemia.ai/courses/llm_inside_and_out/positional_encoding) lesson on Codemia.

**What you will build**: the sinusoidal positional encoding from "Attention Is All You Need", then Rotary Position Embedding (RoPE) from Su et al. Visualize both as heatmaps, then show the defining property of RoPE: after a rotation, the dot product between two positional-encoded vectors depends only on their *relative* distance.

**Karpathy companion**: nanoGPT uses learned absolute positions by default, but this notebook follows the original paper's sinusoidal scheme and the modern RoPE scheme used by LLaMA, Qwen, etc.

**Time**: ~30-45 minutes

**GPU required**: no

## Setup

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)

## Concept recap

Self-attention is permutation-invariant: shuffle the tokens and you get the same set of outputs (just shuffled). So the model needs some way to know "token at position 3" from "token at position 7". Two main approaches:

- **Sinusoidal (absolute)**: add a fixed vector of sines and cosines to each token embedding. Different positions get different vectors; similar positions get similar vectors.
- **RoPE (rotary, relative)**: rotate the query and key vectors in 2D subspaces by an angle proportional to position. The magic: after the rotation, the dot product `q_m . k_n` depends only on `m - n`, not on absolute positions. That means the model gets relative position for free from the attention operation itself.

Modern LLMs (LLaMA, Qwen, DeepSeek, Gemma) all use RoPE. It extrapolates better to longer sequences than learned absolute positions.

## Implementation: sinusoidal positional encoding

In [ ]:
def sinusoidal_pe(seq_len, d_model):
    # Even dimensions are sines, odd dimensions are cosines.
    # Frequencies drop geometrically from 1 (fastest) to 1/10000 (slowest)
    # so different dimensions encode position at different "timescales".
    pe = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, None]  # (seq_len, 1)
    # 2i / d_model ranges from 0 to 1-2/d; 10000^(that) gives the wavelengths.
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

In [ ]:
seq_len = 64
d_model = 64
pe = sinusoidal_pe(seq_len, d_model)
print('pe shape:', pe.shape)

# Values live in [-1, 1] because they're sines/cosines.
print(f'min={pe.min():.3f}  max={pe.max():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pe, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('embedding dimension')
ax.set_ylabel('position')
ax.set_title('Sinusoidal positional encoding')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()
# Notice the vertical stripes on the left (high-frequency dims, change fast with position)
# and the smooth gradients on the right (low-frequency dims, change slowly).
# The combination uniquely identifies every position.

## Nearby positions have similar encodings

A sanity check: if positional encoding is useful, two positions that are close should have similar PE vectors (high cosine similarity), and distant positions should have lower similarity.

In [ ]:
# Cosine similarity matrix between all pairs of positions.
pe_norm = pe / np.linalg.norm(pe, axis=-1, keepdims=True)
sim = pe_norm @ pe_norm.T

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(sim, cmap='viridis')
ax.set_xlabel('position')
ax.set_ylabel('position')
ax.set_title('Cosine similarity between PE vectors')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()
# The diagonal is 1 (self-similarity). The band around the diagonal fades with distance --
# but the pattern is absolute, not relative. Positions 0 and 10 look different from 30 and 40
# even though they have the same gap. RoPE fixes exactly this.

## Implementation: RoPE (Rotary Position Embedding)

In [ ]:
def rope_freqs(d_head, base=10000.0):
    # Same geometric schedule as sinusoidal: each 2-dim pair gets its own frequency.
    # d_head must be even because we operate on pairs of dimensions.
    assert d_head % 2 == 0
    return 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))

def apply_rope(x, freqs):
    # x: (seq_len, d_head). Interpret consecutive dim pairs as 2D vectors and rotate
    # each pair by an angle = position * freq[pair_index].
    seq_len, d_head = x.shape
    positions = torch.arange(seq_len).float().unsqueeze(-1)  # (seq_len, 1)
    # angles[t, i] = how much to rotate pair i at position t.
    angles = positions * freqs.unsqueeze(0)                   # (seq_len, d_head/2)
    cos = angles.cos()
    sin = angles.sin()
    x1 = x[:, 0::2]  # even indices  (the "x" of each 2D pair)
    x2 = x[:, 1::2]  # odd indices   (the "y" of each 2D pair)
    # Standard 2D rotation: (x, y) -> (x cos - y sin, x sin + y cos)
    rx = x1 * cos - x2 * sin
    ry = x1 * sin + x2 * cos
    # Re-interleave back to (..., d_head).
    out = torch.stack([rx, ry], dim=-1).reshape(seq_len, d_head)
    return out

## Visualize RoPE

We apply RoPE to a constant input vector (all ones). Now the pattern at each position is purely the rotation angle, so we can literally see the "clock" each dim-pair makes.

In [ ]:
d_head = 64
freqs = rope_freqs(d_head)
x_const = torch.ones(seq_len, d_head)
x_roped = apply_rope(x_const, freqs).numpy()

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(x_roped, aspect='auto', cmap='RdBu_r', vmin=-1.5, vmax=1.5)
ax.set_xlabel('embedding dimension (pairs of cos/sin)')
ax.set_ylabel('position')
ax.set_title('RoPE applied to a constant vector')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## The defining property: relative distance shows up in the dot product

Here is the whole reason RoPE exists. Pick two random query and key vectors, apply RoPE at a bunch of different absolute positions but keep the *gap* between them fixed, and compute the dot product. It should be constant in absolute position and depend only on the gap.

In [ ]:
# One random query and one random key; identical across trials so variation comes from RoPE alone.
q = torch.randn(1, d_head)
k = torch.randn(1, d_head)

def rope_dot(q_pos, k_pos):
    # Build (max(q_pos,k_pos)+1, d_head), apply RoPE, take the requested rows.
    n = max(q_pos, k_pos) + 1
    qseq = q.expand(n, d_head).contiguous()
    kseq = k.expand(n, d_head).contiguous()
    qr = apply_rope(qseq, freqs)[q_pos]
    kr = apply_rope(kseq, freqs)[k_pos]
    return (qr * kr).sum().item()

# Fixed gap of 5 tokens, absolute position varying from 0 to 30.
gap = 5
print(f"Dot product q . k with RoPE, gap = {gap}:")
for start in range(0, 31, 5):
    d = rope_dot(start + gap, start)
    print(f"  pos {start+gap:3d} vs pos {start:3d}: {d:+.4f}")
print()
print("All values are identical -- relative distance is the only thing that matters.")

In [ ]:
# Sweep the gap from 0 to 32 and plot the dot product. This is what the attention
# score "sees" as a function of relative position, for our one random (q, k) pair.
gaps = list(range(0, 33))
dots = [rope_dot(50 + g, 50) for g in gaps]

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(gaps, dots, marker='.')
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('relative position (q_pos - k_pos)')
ax.set_ylabel('q . k after RoPE')
ax.set_title('RoPE: attention score is a function of relative position only')
plt.tight_layout()
plt.show()
print("Different (q, k) pairs will give different curves, but all of them depend only on the gap.")

## Variation 1: what happens if we extrapolate past the training length?

Sinusoidal and RoPE both extrapolate, but their behavior is different. Sinusoidal produces encodings for any position by formula. RoPE's rotation is well-defined at any position, but the model's *attention scores* at very large gaps start looking like random phases, which is why projects like YaRN and NTK-aware RoPE retune the frequency base. Let's visualize the attention at huge relative distances.

In [ ]:
# Same q, k; sweep gap into the thousands, well past the 64 we built the demo for.
big_gaps = list(range(0, 2048, 64))

def rope_dot_big(q_pos, k_pos):
    n = max(q_pos, k_pos) + 1
    positions = torch.arange(n).float().unsqueeze(-1)
    angles = positions * freqs.unsqueeze(0)
    cos = angles.cos()
    sin = angles.sin()
    qs = q.expand(n, d_head).contiguous()
    ks = k.expand(n, d_head).contiguous()
    def apply(x):
        x1 = x[:, 0::2]; x2 = x[:, 1::2]
        rx = x1 * cos - x2 * sin
        ry = x1 * sin + x2 * cos
        return torch.stack([rx, ry], dim=-1).reshape(n, d_head)
    return (apply(qs)[q_pos] * apply(ks)[k_pos]).sum().item()

long_dots = [rope_dot_big(g, 0) for g in big_gaps]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(big_gaps, long_dots, marker='.')
ax.set_xlabel('relative position')
ax.set_ylabel('q . k after RoPE')
ax.set_title('RoPE attention score at long range (same q, k)')
plt.tight_layout()
plt.show()
# Notice the oscillations decay toward zero. Far-away tokens dot-product to ~0 on average,
# which is the "effective context length" intuition behind RoPE extrapolation studies.

## Variation 2: change the RoPE base

The `10000` base in `rope_freqs` is a hyperparameter. LLaMA uses 10000; Code Llama bumped it to 1,000,000 to reach longer contexts. Larger base -> slower rotation -> the model can distinguish longer distances before the phases wrap.

In [ ]:
for base in [1000, 10000, 100000, 1000000]:
    freqs_base = rope_freqs(d_head, base=base)
    # How fast does the slowest dimension-pair rotate? Period = 2*pi / freq.
    slowest_period = 2 * np.pi / freqs_base[-1].item()
    print(f"base={base:>8d}: slowest period ~= {slowest_period:10.0f} tokens")
print()
print("Bigger base => bigger period on the slow end => more effective distance before wrap.")

## What to try next

- Implement ALiBi (Press et al., 2021): instead of rotating, add a linear bias `-|i - j|` times a per-head slope to the attention logits. It's even simpler than RoPE and also extrapolates.
- Add RoPE to the `CausalSelfAttention` from `transformer_block.ipynb` and train a tiny model with and without it on the same data.
- Compute the angle of the slowest RoPE dimension-pair at position 8192 for base=10000 vs base=1000000. That tells you why long-context models rebase.